# Contrastive Fine-Tuning: Merging Clean & Typo Representations
*Optimizes LoRA parameters to pull paired clean/typo activations together.*

In [ ]:
!pip install -q torch transformers accelerate peft scikit-learn tqdm

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import json

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

In [ ]:
# Load dataset (assumes df is already loaded from previous notebook)
# If not, use upload cell
DATASET_PATH = "typosquat_dataset_full/typosquat_tool_calls.jsonl"
df_all = pd.DataFrame([json.loads(line) for line in open(DATASET_PATH)])
df = df_all[df_all['is_adversarial'] == True].copy()
print(f"Loaded {len(df)} adversarial entries")

df_train, df_test = train_test_split(df, test_size=0.2, random_state=SEED)
print(f"Train: {len(df_train)}, Test: {len(df_test)}")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
class ContrastiveDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=64):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pairs = [(row['clean_command'], row['typo_command']) for _, row in df.iterrows()]
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        c, t = self.pairs[idx]
        enc_c = self.tokenizer(c, max_length=self.max_length, truncation=True, padding='max_length', return_tensors='pt')
        enc_t = self.tokenizer(t, max_length=self.max_length, truncation=True, padding='max_length', return_tensors='pt')
        return {
            'input_ids_c': enc_c['input_ids'].squeeze(0).to(DEVICE),
            'attn_c': enc_c['attention_mask'].squeeze(0).to(DEVICE),
            'input_ids_t': enc_t['input_ids'].squeeze(0).to(DEVICE),
            'attn_t': enc_t['attention_mask'].squeeze(0).to(DEVICE)
        }

subset_df = df_train.sample(n=500, random_state=SEED)
dataset = ContrastiveDataset(subset_df, tokenizer, max_length=64)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
epochs = 3
model.train()
for epoch in range(epochs):
    epoch_loss = 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        out_c = model(input_ids=batch['input_ids_c'], attention_mask=batch['attn_c'], output_hidden_states=True)
        h_c = out_c.hidden_states[14].mean(dim=1)
        out_t = model(input_ids=batch['input_ids_t'], attention_mask=batch['attn_t'], output_hidden_states=True)
        h_t = out_t.hidden_states[14].mean(dim=1)
        loss = 1 - F.cosine_similarity(h_c, h_t, dim=-1).mean()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(dataloader):.4f}")

model = model.merge_and_unload()

In [ ]:
def extract_features(model, tokenizer, commands, max_length=64):
    model.eval()
    feats = []
    with torch.no_grad():
        for i in range(0, len(commands), 4):
            batch = tokenizer(commands[i:i+4], padding=True, truncation=True, max_length=max_length, return_tensors='pt').to(DEVICE)
            out = model(**batch, output_hidden_states=True)
            for j in range(len(batch['input_ids'])):
                feats.append(out.hidden_states[14][j, -1, :].float().cpu().numpy())
    return np.array(feats)

test_texts, test_labels = [], []
for _, row in df_test.iterrows():
    test_texts.append(row['clean_command']); test_labels.append(0)
    test_texts.append(row['typo_command']); test_labels.append(1)

X_test = extract_features(model, tokenizer, test_texts)
y_test = np.array(test_labels)

probe = LogisticRegression(max_iter=2000).fit(X_test, y_test)
auc = roc_auc_score(y_test, probe.predict_proba(X_test)[:, 1])
print(f"Post-Contrastive AUC: {auc:.4f}")
print(f"ΔAUC from baseline (0.9985): {0.9985 - auc:.4f}")